![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 03 | Hypotheses and Visualisation

*Project 1 — SQL: From Data to Insight*

**Team: Regina Khamatnurova**
**Dataset:https://cpds-data.org/data/.**

---

**This notebook is your report.** It is read on its own, by someone who has not seen notebooks 01 and 02, so it needs the prose as much as the code: the question, the evidence, and what you concluded. Keep the messy exploration in the other two.

> **Done when:** someone who has never met your project can read this top to bottom and understand what you asked, what the data said, and how confident they should be.

The sections below are a suggested route, not a form to fill in. Add sections, drop them, reorder them — what is graded is the analysis, not whether you kept the scaffold.

# Hypothesis Testing and Visualization

This notebook uses SQL queries to investigate the relationship between
government political composition and economic outcomes across 36 countries
between 2000 and 2019.

The analysis focuses on descriptive associations rather than causal effects.

---
## 0. Setup

In [2]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")


import sqlite3
from pathlib import Path

---
## 1. The question

What you set out to find out, and why it matters. Your research questions from notebook 01, with the hypothesis for each.

´´´
1. GDP growth and government composition
Research question:
How does average real GDP growth differ across government political compositions between 2000 and 2019?
Expectation:
I expect to observe some differences in average GDP growth between government composition categories, although I do not expect political composition alone to explain these differences because economic growth is influenced by many other factors.

2. Unemployment and government composition
Research question:
How does the unemployment rate differ across government political compositions between 2000 and 2019?
Expectation:
I expect unemployment rates to vary across government composition categories. However, I expect substantial variation within each category because unemployment is also influenced by country-specific conditions and broader economic cycles.

3. Inflation 
Research question:
How does inflation differ across government political compositions between 2000 and 2019?
Expectation:
I expect inflation to show differences across government composition categories, but I also expect considerable variation between countries and years.
These work well because you could genuinely be wrong. For example, your SQL analysis might show virtually no meaningful difference in average GDP growth between the categories. That's still a valid result.
´´´

---
## 2. The data

Where it came from, what one row is, and what your database looks like. Show the ERD.

The data for this project came from https://cpds-data.org/data/. One row represents different economical factors for 36 countries for each year for the time periog 2000 - 2019

In [3]:
database_path = Path("../data/project.db")

connection = sqlite3.connect(database_path)
connection.execute("PRAGMA foreign_keys = ON;")

In [4]:
pd.read_sql(
    """
    SELECT *
    FROM country_year
    LIMIT 5;
    """,
    connection
)

,observation_id,country_id,government_type_id,year,left_percentage,centre_percentage,right_percentage,gdp_growth,unemployment,inflation
0,1,1,1,2000,0.0,0.0,100.0,3.386844,6.3,4.457435
1,2,1,1,2001,0.0,0.0,100.0,2.714849,6.7,4.407135
2,3,1,1,2002,0.0,0.0,100.0,4.010583,6.4,2.981575
3,4,1,1,2003,0.0,0.0,100.0,3.119654,5.9,2.732596
4,5,1,1,2004,0.0,0.0,100.0,3.964010,5.4,2.343255


---
## 3. Analysis

One section per question: the query, the result, and a sentence saying what it means. Five queries minimum across the notebook, and the aggregation belongs in SQL rather than in pandas afterwards. Keep the queries in `sql/queries.sql` — this notebook runs them.

### Research Question 1: GDP Growth

**Question:** How does average real GDP growth differ across government
political compositions between 2000 and 2019?

**Expectation:** Average GDP growth may differ between government composition
categories, but government composition alone is unlikely to explain the
variation in economic growth.

In [9]:
query_gdp = """
SELECT
    gt.government_type,
    ROUND(AVG(cy.gdp_growth), 2) AS avg_gdp_growth,
    COUNT(*) AS observations
FROM country_year AS cy
INNER JOIN government_types AS gt
    ON cy.government_type_id = gt.government_type_id
GROUP BY gt.government_type_id, gt.government_type
ORDER BY gt.government_type_id;
"""

gdp_by_government = pd.read_sql(query_gdp, connection)

gdp_by_government

,government_type,avg_gdp_growth,observations
0,Hegemony of right-wing and centre parties,2.50,294
1,Dominance of right-wing and centre parties,2.67,129
2,Balance between left and right,2.44,131
3,Dominance of social-democratic and other left ...,1.93,89
4,Hegemony of social-democratic and other left p...,2.75,77


#### Finding

Average real GDP growth varies across the five government composition
categories. However, there is no clear linear pattern across the Schmidt
Index categories.

The number of observations also differs substantially between categories,
ranging from 77 to 294 country-year observations. Therefore, these averages
should be interpreted as descriptive associations rather than evidence that
government composition causes differences in GDP growth.

---

### Research Question 2: Unemployment

**Question:** How does average unemployment differ across government
political compositions between 2000 and 2019?

**Expectation:** Unemployment may vary across government composition
categories, while substantial differences may also reflect country-specific
conditions and economic cycles.

In [10]:
query_unemployment = """
SELECT
    gt.government_type,
    ROUND(AVG(cy.unemployment), 2) AS avg_unemployment,
    COUNT(*) AS observations
FROM country_year AS cy
INNER JOIN government_types AS gt
    ON cy.government_type_id = gt.government_type_id
GROUP BY gt.government_type_id, gt.government_type
ORDER BY gt.government_type_id;
"""

unemployment_by_government = pd.read_sql(
    query_unemployment,
    connection
)

unemployment_by_government

,government_type,avg_unemployment,observations
0,Hegemony of right-wing and centre parties,7.57,294
1,Dominance of right-wing and centre parties,8.59,129
2,Balance between left and right,7.56,131
3,Dominance of social-democratic and other left ...,9.56,89
4,Hegemony of social-democratic and other left p...,7.46,77


#### Finding

Average unemployment varies across government composition categories, ranging
from 7.46% to 9.56% in the selected period.

As with GDP growth, there is no clear linear pattern across the five Schmidt
Index categories. The number of observations differs substantially between
categories, and the results combine multiple countries and years. Therefore,
the differences should be interpreted as descriptive associations rather than
causal effects of government composition.

---

### Research Question 2: Inflation

In [11]:
query_inflation = """
SELECT
    gt.government_type,
    ROUND(AVG(cy.inflation), 2) AS avg_inflation,
    COUNT(*) AS observations
FROM country_year AS cy
INNER JOIN government_types AS gt
    ON cy.government_type_id = gt.government_type_id
GROUP BY gt.government_type_id, gt.government_type
ORDER BY gt.government_type_id;
"""

inflation_by_government = pd.read_sql(
    query_inflation,
    connection
)

inflation_by_government

,government_type,avg_inflation,observations
0,Hegemony of right-wing and centre parties,2.18,294
1,Dominance of right-wing and centre parties,2.17,129
2,Balance between left and right,2.64,131
3,Dominance of social-democratic and other left ...,2.44,89
4,Hegemony of social-democratic and other left p...,2.82,77


#### Finding

Average inflation varies moderately across government composition categories,
ranging from 2.17% to 2.82%.

As with GDP growth and unemployment, the averages do not show a clear linear
pattern across the five Schmidt Index categories. These results describe
associations in the pooled country-year data and should not be interpreted
as causal effects of government composition.

---

In [12]:
query_country_gdp = """
SELECT
    c.country_name,
    gt.government_type,
    ROUND(AVG(cy.gdp_growth), 2) AS avg_gdp_growth,
    COUNT(*) AS observations
FROM country_year AS cy
INNER JOIN countries AS c
    ON cy.country_id = c.country_id
INNER JOIN government_types AS gt
    ON cy.government_type_id = gt.government_type_id
GROUP BY
    c.country_name,
    gt.government_type_id,
    gt.government_type
HAVING COUNT(*) >= 3
ORDER BY
    c.country_name,
    gt.government_type_id;
"""

country_gdp = pd.read_sql(
    query_country_gdp,
    connection
)

country_gdp

,country_name,government_type,avg_gdp_growth,observations
0,Australia,Hegemony of right-wing and centre parties,2.90,13
1,Australia,Hegemony of social-democratic and other left p...,2.71,5
2,Austria,Hegemony of right-wing and centre parties,2.04,8
3,Austria,Balance between left and right,1.22,11
4,Belgium,Hegemony of right-wing and centre parties,1.69,5
...,...,...,...,...
88,Sweden,Hegemony of social-democratic and other left p...,2.72,11
89,Switzerland,Dominance of right-wing and centre parties,1.95,20
90,USA,Hegemony of right-wing and centre parties,2.16,20
91,United Kingdom,Hegemony of right-wing and centre parties,1.94,9


#### Finding

The country-level results show substantial variation in average GDP growth
within the same government composition categories.

This suggests that the overall averages by government type may hide important
country-specific differences. The query includes only country-government type
combinations with at least three observations to avoid comparisons based on
very small groups.

---

In [13]:
query_above_average = """
SELECT
    c.country_name,
    gt.government_type,
    ROUND(AVG(cy.gdp_growth), 2) AS avg_gdp_growth,
    COUNT(*) AS observations
FROM country_year AS cy
INNER JOIN countries AS c
    ON cy.country_id = c.country_id
INNER JOIN government_types AS gt
    ON cy.government_type_id = gt.government_type_id
GROUP BY
    c.country_name,
    gt.government_type_id,
    gt.government_type
HAVING
    COUNT(*) >= 3
    AND AVG(cy.gdp_growth) > (
        SELECT AVG(gdp_growth)
        FROM country_year
    )
ORDER BY
    c.country_name,
    gt.government_type_id;
"""

above_average_gdp = pd.read_sql(
    query_above_average,
    connection
)

above_average_gdp

,country_name,government_type,avg_gdp_growth,observations
0,Australia,Hegemony of right-wing and centre parties,2.90,13
1,Australia,Hegemony of social-democratic and other left p...,2.71,5
2,Bulgaria,Hegemony of right-wing and centre parties,2.57,6
3,Bulgaria,Dominance of right-wing and centre parties,3.81,9
4,Bulgaria,Balance between left and right,4.00,5
5,Croatia,Balance between left and right,4.39,4
6,Cyprus,Hegemony of right-wing and centre parties,4.43,9
7,Cyprus,Balance between left and right,4.03,7
8,Czech Republic,Balance between left and right,4.53,8
9,Czech Republic,Dominance of social-democratic and other left ...,2.88,3


#### Finding

46 country-government type combinations with at least three observations had
average GDP growth above the overall average for the 2000–2019 dataset.

These combinations occur across different government composition categories
and countries. Together with the previous country-level analysis, this
suggests that country-specific variation is important when interpreting the
overall averages by government composition.

The results remain descriptive and do not establish a causal relationship
between government composition and economic growth.

---
## 4. Visualisation

At least two charts that carry an argument. Labelled axes, a title that states the finding, and a chart type that suits the data.

---
## 5. Conclusions

Answer each question. Say whether your hypothesis held. Being wrong is a fine result, as long as you say so and show the evidence.

---
## 6. Limitations and next steps

What this data cannot tell you, and what you would do with another week. Knowing the limits of your own analysis is part of the grade.